# Saf v3 ve tam yüzeyde 20 mm iri taş düzeltmesi

Bu notebook 24 toprakta, yalnızca 20 mm elek tahminini değiştirmenin etkisini ölçer. Önce H615’in iki fotoğrafında otomatik ve elle çizilen taş sınırlarını karşılaştırır. İki kontrol geçerse 127 eğitim çekimini işler ve 66 saklı tahmini değerlendirir.

**Kaggle’da çalıştırma.** Notebook’u içe aktar, **Add Input** ile **Soil Grain Size from Photos** yarışmasını ekle. Ayarlardan **GPU** hızlandırıcısını ve **Internet** erişimini açıp **Run All** seç. Cellpose 4.0.6 ve ImageGrains’in yaklaşık 1,2 GB ağırlık dosyası ilk çalıştırmada indirilir. Bu işlem var olan ağırlıklarla görüntü ölçümü yapar. Yeni sinir ağı eğitimi veya Kaggle gönderimi içermez.

Son hücre sonuçları dört sınama için gösterir. İndirilecek özet dosyası `/kaggle/working/soil20/ozet.csv`. Çekim alanları, iki kontrolün görseli ve numune bazında sonuçlar aynı klasördedir. Düşük EMD daha iyidir. Pozitif kazanç, düzeltmenin hatayı azalttığı anlamına gelir.

[Yarışma verileri](https://www.kaggle.com/competitions/soil-grain-size-from-photos/data) · [ImageGrains ağırlıkları](https://zenodo.org/records/15728186) · [Arşivlenmiş saf v3 sonuçları](https://github.com/MustafaEmreGulum/soil-grain-size-from-photos/blob/a15272c/outputs/kutle_payi_kontrol_sonuclari.csv)

In [ ]:
import sys
import subprocess
from importlib import metadata
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Önce Kaggle ayarlarında GPU hızlandırıcısını açıp oturumu yeniden başlat.")
print("GPU", torch.cuda.get_device_name(0))
try:
    installed = metadata.version("cellpose")
except metadata.PackageNotFoundError:
    installed = None
if installed != "4.0.6":
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","cellpose==4.0.6"])

In [ ]:
import hashlib
import os
import shutil
import zipfile
from io import StringIO
from urllib.request import urlopen
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image, ImageOps
from skimage.measure import regionprops
from cellpose import models

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/soil20")
TEMP = Path("/kaggle/temp/soil20")
OUT.mkdir(exist_ok=True,parents=True)
TEMP.mkdir(exist_ok=True,parents=True)
PPM = 4.5
PHONES = {"Motorola_Edge_60_fusion":(4096,12.465),"Motorola_Edge":(4000,11.492),
          "Samsung_A52":(9248,26.33)}
COLS = ["0.002","0.0063","0.02","0.063","0.2","0.63","2","6.3","20","63","200"]
WIDTHS = np.diff(np.log10(np.array(COLS,dtype=float)))

### Kaydedilmiş karşılaştırma verileri

Aşağıdaki iki tabloda, önceki deneyde belirlenmiş tepsi içi kadrajlar ve yalnız saklı toprak için üretilmiş saf v3 tahminleri bulunur. Laboratuvar eğrileri Kaggle yarışma verisinden okunur. H615’in elle çizilmiş iki sınırı, otomatik maske üretilmeden önce milimetre cinsinden kaydedilmiştir.

In [ ]:
boxes = pd.read_csv(StringIO("""file,sample_id,phone,width,height,left,top,right,bottom
Motorola_Edge_60_fusion_H374_01.jpg,H374,Motorola_Edge_60_fusion,4096,2304,59,40,4037,2264
Motorola_Edge_60_fusion_H374_02.jpg,H374,Motorola_Edge_60_fusion,4096,2304,59,40,4037,2264
Motorola_Edge_60_fusion_H374_03.jpg,H374,Motorola_Edge_60_fusion,4096,2304,59,40,4037,2264
Motorola_Edge_F827_01.jpg,F827,Motorola_Edge,1600,720,157,12,1565,708
Motorola_Edge_F827_02.jpg,F827,Motorola_Edge,1600,720,157,12,1569,708
Motorola_Edge_G190_01.jpg,G190,Motorola_Edge,1600,720,153,12,1527,708
Motorola_Edge_G190_02.jpg,G190,Motorola_Edge,1600,720,157,12,1577,708
Motorola_Edge_H030_01.jpg,H030,Motorola_Edge,1600,720,23,12,1428,708
Motorola_Edge_H030_02.jpg,H030,Motorola_Edge,1600,720,65,12,1432,708
Motorola_Edge_H030_03.jpg,H030,Motorola_Edge,1600,720,172,12,1577,708
Motorola_Edge_H030_04.jpg,H030,Motorola_Edge,1600,720,160,12,1535,708
Motorola_Edge_H037_01.jpg,H037,Motorola_Edge,1600,720,145,12,1523,708
Motorola_Edge_H037_02.jpg,H037,Motorola_Edge,1600,720,73,12,1455,708
Motorola_Edge_H037_03.jpg,H037,Motorola_Edge,1600,720,138,12,1520,708
Motorola_Edge_H038_01.jpg,H038,Motorola_Edge,1600,720,27,12,1432,708
Motorola_Edge_H038_02.jpg,H038,Motorola_Edge,1600,720,23,12,1420,708
Motorola_Edge_H038_03.jpg,H038,Motorola_Edge,1600,720,172,12,1573,708
Motorola_Edge_H126_01.jpg,H126,Motorola_Edge,1600,720,100,12,1474,708
Motorola_Edge_H126_02.jpg,H126,Motorola_Edge,1600,720,31,12,1436,708
Motorola_Edge_H126_03.jpg,H126,Motorola_Edge,1600,720,80,12,1462,708
Motorola_Edge_H181_01.jpg,H181,Motorola_Edge,1600,720,54,12,1417,708
Motorola_Edge_H181_02.jpg,H181,Motorola_Edge,1600,720,31,12,1447,708
Motorola_Edge_H181_03.jpg,H181,Motorola_Edge,1600,720,149,27,1531,708
Motorola_Edge_H181_04.jpg,H181,Motorola_Edge,1600,720,96,12,1474,708
Motorola_Edge_H183_01.jpg,H183,Motorola_Edge,1600,720,126,12,1546,708
Motorola_Edge_H183_02.jpg,H183,Motorola_Edge,1600,720,103,12,1462,708
Motorola_Edge_H183_03.jpg,H183,Motorola_Edge,1600,720,145,12,1546,708
Motorola_Edge_H183_04.jpg,H183,Motorola_Edge,1600,720,172,12,1546,693
Motorola_Edge_H366_01.jpg,H366,Motorola_Edge,1600,720,172,12,1577,708
Motorola_Edge_H366_02.jpg,H366,Motorola_Edge,1600,720,149,12,1573,708
Motorola_Edge_H366_03.jpg,H366,Motorola_Edge,1600,720,145,12,1561,708
Motorola_Edge_H366_04.jpg,H366,Motorola_Edge,1600,720,23,12,1447,708
Motorola_Edge_H367_01.jpg,H367,Motorola_Edge,1600,720,35,12,1447,708
Motorola_Edge_H367_02.jpg,H367,Motorola_Edge,1600,720,35,12,1447,708
Motorola_Edge_H367_03.jpg,H367,Motorola_Edge,1600,720,149,12,1565,708
Motorola_Edge_H368_01.jpg,H368,Motorola_Edge,1600,720,187,12,1497,708
Motorola_Edge_H368_02.jpg,H368,Motorola_Edge,1600,720,153,12,1531,693
Motorola_Edge_H368_03.jpg,H368,Motorola_Edge,1600,720,134,12,1539,708
Motorola_Edge_H368_04.jpg,H368,Motorola_Edge,1600,720,134,12,1504,708
Motorola_Edge_H371_01.jpg,H371,Motorola_Edge,1600,720,84,12,1451,708
Motorola_Edge_H371_02.jpg,H371,Motorola_Edge,1600,720,145,12,1512,708
Motorola_Edge_H371_03.jpg,H371,Motorola_Edge,1600,720,157,12,1577,708
Motorola_Edge_H372_01.jpg,H372,Motorola_Edge,1600,720,145,27,1375,708
Motorola_Edge_H372_02.jpg,H372,Motorola_Edge,1600,720,31,27,1459,708
Motorola_Edge_H372_03.jpg,H372,Motorola_Edge,1600,720,31,12,1424,693
Motorola_Edge_H372_04.jpg,H372,Motorola_Edge,1600,720,172,12,1577,708
Motorola_Edge_H405_01.jpg,H405,Motorola_Edge,1600,720,168,12,1577,708
Motorola_Edge_H405_02.jpg,H405,Motorola_Edge,1600,720,69,12,1451,708
Motorola_Edge_H405_03.jpg,H405,Motorola_Edge,1600,720,65,12,1447,708
Motorola_Edge_H493_01.jpg,H493,Motorola_Edge,1600,720,157,12,1542,708
Motorola_Edge_H493_02.jpg,H493,Motorola_Edge,1600,720,80,12,1455,708
Motorola_Edge_H493_03.jpg,H493,Motorola_Edge,1600,720,157,12,1531,708
Motorola_Edge_H516_01.jpg,H516,Motorola_Edge,1600,720,157,12,1542,708
Motorola_Edge_H516_02.jpg,H516,Motorola_Edge,1600,720,61,12,1455,708
Motorola_Edge_H516_03.jpg,H516,Motorola_Edge,1600,720,23,12,1455,708
Motorola_Edge_H549_01.jpg,H549,Motorola_Edge,1600,720,69,12,1443,708
Motorola_Edge_H549_02.jpg,H549,Motorola_Edge,1600,720,157,12,1535,708
Motorola_Edge_H549_03.jpg,H549,Motorola_Edge,1600,720,157,12,1539,708
Motorola_Edge_H615_01.jpg,H615,Motorola_Edge,1600,720,111,16,1520,708
Motorola_Edge_H615_02.jpg,H615,Motorola_Edge,1600,720,149,12,1577,708
Motorola_Edge_H616_01.jpg,H616,Motorola_Edge,1600,720,73,12,1447,708
Motorola_Edge_H616_02.jpg,H616,Motorola_Edge,1600,720,65,12,1443,708
Motorola_Edge_H616_03.jpg,H616,Motorola_Edge,1600,720,61,12,1436,708
Motorola_Edge_H617_01.jpg,H617,Motorola_Edge,1600,720,23,12,1428,708
Motorola_Edge_H617_02.jpg,H617,Motorola_Edge,1600,720,172,12,1577,708
Motorola_Edge_H617_03.jpg,H617,Motorola_Edge,1600,720,23,12,1428,708
Motorola_Edge_H666_01.jpg,H666,Motorola_Edge,1600,720,160,12,1573,708
Motorola_Edge_H666_02.jpg,H666,Motorola_Edge,1600,720,65,12,1447,708
Motorola_Edge_H666_03.jpg,H666,Motorola_Edge,1600,720,69,12,1447,708
Motorola_Edge_H668_01.jpg,H668,Motorola_Edge,1600,720,138,12,1516,708
Motorola_Edge_H668_02.jpg,H668,Motorola_Edge,1600,720,134,12,1508,708
Motorola_Edge_H668_03.jpg,H668,Motorola_Edge,1600,720,80,12,1459,708
Samsung_A52_F827_01.jpg,F827,Samsung_A52,1599,1200,138,229,1511,1123
Samsung_A52_F827_02.jpg,F827,Samsung_A52,1599,1200,149,206,1530,1104
Samsung_A52_F827_03.jpg,F827,Samsung_A52,1599,1200,164,183,1572,1074
Samsung_A52_G190_01.jpg,G190,Samsung_A52,1599,1200,141,229,1522,1116
Samsung_A52_G190_02.jpg,G190,Samsung_A52,1599,1200,160,233,1568,1116
Samsung_A52_G190_03.jpg,G190,Samsung_A52,1599,1200,107,237,1488,1135
Samsung_A52_H030_01.jpg,H030,Samsung_A52,1599,1200,126,225,1500,1108
Samsung_A52_H030_02.jpg,H030,Samsung_A52,1599,1200,130,206,1503,1093
Samsung_A52_H030_03.jpg,H030,Samsung_A52,1599,1200,130,233,1507,1120
Samsung_A52_H037_01.jpg,H037,Samsung_A52,1599,1200,103,218,1484,1123
Samsung_A52_H037_02.jpg,H037,Samsung_A52,1599,1200,111,218,1492,1120
Samsung_A52_H037_03.jpg,H037,Samsung_A52,1599,1200,115,214,1492,1100
Samsung_A52_H038_01.jpg,H038,Samsung_A52,1599,1200,115,229,1492,1120
Samsung_A52_H038_02.jpg,H038,Samsung_A52,1599,1200,141,202,1519,1089
Samsung_A52_H038_03.jpg,H038,Samsung_A52,1599,1200,138,233,1519,1127
Samsung_A52_H038_04.jpg,H038,Samsung_A52,1599,1200,119,176,1500,1066
Samsung_A52_H038_05.jpg,H038,Samsung_A52,1599,1200,122,229,1503,1120
Samsung_A52_H126_01.jpg,H126,Samsung_A52,1599,1200,126,202,1503,1097
Samsung_A52_H181_01.jpg,H181,Samsung_A52,1599,1200,111,229,1503,1131
Samsung_A52_H181_02.jpg,H181,Samsung_A52,1599,1200,119,187,1515,1089
Samsung_A52_H183_01.jpg,H183,Samsung_A52,1599,1200,126,210,1519,1108
Samsung_A52_H183_02.jpg,H183,Samsung_A52,1599,1200,134,195,1511,1089
Samsung_A52_H183_03.jpg,H183,Samsung_A52,1599,1200,138,191,1519,1097
Samsung_A52_H367_01.jpg,H367,Samsung_A52,1599,1200,99,210,1496,1123
Samsung_A52_H368_01.jpg,H368,Samsung_A52,1599,1200,115,199,1522,1108
Samsung_A52_H371_01.jpg,H371,Samsung_A52,1599,1200,134,206,1534,1112
Samsung_A52_H371_02.jpg,H371,Samsung_A52,1599,1200,111,214,1534,1135
Samsung_A52_H371_03.jpg,H371,Samsung_A52,1599,1200,103,122,1515,1112
Samsung_A52_H371_04.jpg,H371,Samsung_A52,1599,1200,99,126,1519,1131
Samsung_A52_H372_01.jpg,H372,Samsung_A52,1599,1200,141,218,1572,1112
Samsung_A52_H372_02.jpg,H372,Samsung_A52,1599,1200,153,210,1576,1120
Samsung_A52_H405_01.jpg,H405,Samsung_A52,1599,1200,160,237,1534,1116
Samsung_A52_H405_02.jpg,H405,Samsung_A52,1599,1200,130,229,1507,1127
Samsung_A52_H405_03.jpg,H405,Samsung_A52,1599,1200,126,229,1503,1120
Samsung_A52_H493_01.jpg,H493,Samsung_A52,1599,1200,126,225,1492,1100
Samsung_A52_H493_02.jpg,H493,Samsung_A52,1599,1200,122,233,1500,1123
Samsung_A52_H516_01.jpg,H516,Samsung_A52,1599,1200,107,233,1492,1131
Samsung_A52_H549_01.jpg,H549,Samsung_A52,1599,1200,122,229,1500,1127
Samsung_A52_H549_02.jpg,H549,Samsung_A52,1599,1200,122,240,1492,1135
Samsung_A52_H615_01.jpg,H615,Samsung_A52,1599,1200,111,218,1522,1123
Samsung_A52_H615_02.jpg,H615,Samsung_A52,1599,1200,107,218,1522,1120
Samsung_A52_H615_03.jpg,H615,Samsung_A52,1599,1200,92,218,1507,1123
Samsung_A52_H616_01.jpg,H616,Samsung_A52,1599,1200,119,195,1492,1093
Samsung_A52_H616_02.jpg,H616,Samsung_A52,1599,1200,115,218,1492,1108
Samsung_A52_H617_01.jpg,H617,Samsung_A52,1599,1200,122,237,1503,1135
Samsung_A52_H617_02.jpg,H617,Samsung_A52,1599,1200,96,221,1477,1116
Samsung_A52_H617_03.jpg,H617,Samsung_A52,1599,1200,119,252,1496,1135
Samsung_A52_H637_01.jpg,H637,Samsung_A52,1599,1200,115,221,1507,1120
Samsung_A52_H637_02.jpg,H637,Samsung_A52,1599,1200,130,210,1515,1104
Samsung_A52_H637_03.jpg,H637,Samsung_A52,1599,1200,107,218,1519,1131
Samsung_A52_H666_01.jpg,H666,Samsung_A52,1599,1200,119,252,1496,1135
Samsung_A52_H668_01.jpg,H668,Samsung_A52,1599,1200,122,237,1484,1123
Samsung_A52_H668_02.jpg,H668,Samsung_A52,1599,1200,115,199,1488,1089
Samsung_A52_H668_03.jpg,H668,Samsung_A52,1599,1200,99,218,1484,1108
Samsung_A52_H668_04.jpg,H668,Samsung_A52,1599,1200,126,214,1503,1104
""")).set_index("file")
curves = pd.read_csv(StringIO("""exam,sample_id,emd,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
24,F827,31.429603849,0.465737154,17.519696875,37.966164525,66.846167245,87.082023347,95.46328553,98.229193328,99.502932671,100.0,100.0,100.0
24,G190,17.735623619,0.443948275,8.776092963,22.557032409,43.999198339,74.018363392,92.860396149,97.713044043,99.525379799,100.0,100.0,100.0
24,H030,18.562237238,0.445957762,9.108567432,24.17935727,46.798662098,76.321416453,92.326026515,97.360629576,99.406453796,100.0,100.0,100.0
24,H037,91.44780874,0.442311133,8.724337952,22.279565708,46.548407641,77.431038901,93.343879068,97.749205876,99.503515677,100.0,100.0,100.0
24,H038,119.659860516,0.434213039,6.980031272,18.045923815,38.60408333,69.192852327,90.076642914,96.939113365,99.288497955,100.0,100.0,100.0
24,H126,28.846640434,0.446582525,9.101309677,23.343889526,45.147573664,74.94489148,92.912696184,97.620261355,99.429145212,100.0,100.0,100.0
24,H181,20.282593965,0.389140921,1.626001007,3.931460275,7.341880312,11.806345931,19.667462307,34.651431377,58.838658938,86.508324995,98.840557378,100.0
24,H183,19.585116978,0.386409831,1.57852149,3.753547274,7.0468111,11.468303288,18.851859188,32.862655534,56.270319856,84.159026103,98.127175259,100.0
24,H366,21.750993107,0.348848634,1.472588146,4.133788047,7.865619441,13.455729813,24.262123279,44.35765495,71.812341027,93.335207124,99.645312145,100.0
24,H367,20.444715804,0.346035514,1.452930042,4.063196966,7.766126124,13.147382536,23.256704667,42.265417238,69.129159817,92.26879668,99.601927082,100.0
24,H368,13.665640054,0.321156374,1.262982118,3.347833235,6.568414382,10.731385937,17.270748767,29.60407866,51.357717659,78.846001133,96.807537614,100.0
24,H371,23.831720074,0.317619924,1.096040396,2.899317977,5.826667571,9.658182548,15.643087875,27.049804919,47.834846861,75.582244017,95.937836631,100.0
24,H372,35.335035055,0.305438781,0.988990248,2.647514027,5.300355776,8.668048939,13.798120729,23.218073803,41.181990411,66.965466233,91.924787964,100.0
24,H374,52.404023621,0.363161712,1.683054529,4.90352556,9.416815069,16.932460115,31.133432311,55.421435489,83.390522852,96.900180876,99.889468573,100.0
24,H405,74.213617787,0.436540951,6.715603768,16.498592846,35.037730274,63.23383134,86.972760638,96.258050662,99.065600621,100.0,100.0,100.0
24,H493,56.237506782,0.484079817,33.504409769,60.801775609,83.523294249,93.789977323,97.316879445,99.114961936,100.0,100.0,100.0,100.0
24,H516,44.574924401,0.416906136,2.66743888,7.186848943,13.322860762,25.551145407,46.414583916,74.692425018,93.176579385,98.876493753,100.0,100.0
24,H549,37.314997458,0.447275912,9.296797563,23.577092635,45.52897928,75.331777275,92.153250777,97.25960842,99.291968308,100.0,100.0,100.0
24,H615,24.140041863,0.323610445,1.209794784,3.038303221,5.976950759,9.668384148,15.129289602,25.214996003,43.399760787,69.208235077,92.710634146,100.0
24,H616,60.225591331,0.488021475,11.004734244,26.144442687,48.668137108,77.043596008,92.49538482,97.318635849,99.531325906,100.0,100.0,100.0
24,H617,21.259471312,0.446514578,9.992251334,24.864115921,47.276623617,76.3793443,92.524377617,97.848307633,99.55583061,100.0,100.0,100.0
24,H637,32.675122286,0.347355201,1.387522973,3.897049956,7.590893397,12.535275405,21.943627048,41.40780481,70.176623222,92.8845262,99.621443126,100.0
24,H666,28.230767014,0.452654973,12.623127114,29.813513653,54.519638768,80.90856669,93.795247992,97.778026999,99.40082469,100.0,100.0,100.0
24,H668,36.327571502,0.448639799,9.36987084,24.114238982,46.387678038,75.736279706,92.370082788,97.7370851,99.536683626,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,F827,65.951006173,0.444632131,8.561057013,22.418865572,43.656851763,72.725423045,90.485215318,96.734366713,99.02687364,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,G190,40.427477241,0.412877988,4.433382919,12.635615396,28.135116879,52.818494573,81.490306769,95.367636665,98.840635053,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H030,12.02191632,0.429868954,6.415501772,17.841720181,37.022501751,65.952445714,87.940851315,96.209218012,99.085644816,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H037,66.357163626,0.413395587,4.791150099,13.022450157,29.562652304,57.947966795,85.35576717,95.86120712,98.919890902,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H038,99.562888035,0.408434858,4.344307062,12.226262146,28.282710128,55.091072643,83.270793365,95.278396852,98.812664614,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H126,67.022358254,0.405544453,3.844573951,10.986945384,24.13616547,46.569002451,76.300287536,93.864759017,98.329721315,99.851962184,100.0,100.0
Motorola_Edge -> Samsung_A52,H181,21.449988349,0.33814908,1.167492243,2.960262741,6.213170788,9.971465739,16.433518053,29.612475253,55.537932277,86.572466313,99.178841224,100.0
Motorola_Edge -> Samsung_A52,H183,12.313933341,0.344551125,1.228816152,3.18822302,6.585383481,10.78451032,18.462694014,34.519733318,62.434155221,90.248839094,99.57127563,100.0
Motorola_Edge -> Samsung_A52,H367,17.677549715,0.283649568,0.815271883,2.984042936,6.604145728,11.135947465,19.999973758,38.930620956,68.719356693,93.300911923,99.65066585,100.0
Motorola_Edge -> Samsung_A52,H368,9.962370112,0.241485134,0.490640411,2.370542375,5.422171641,8.817484606,14.244062309,25.295474017,47.782966991,78.170719666,97.360044559,100.0
Motorola_Edge -> Samsung_A52,H371,22.577392744,0.249351619,0.498506895,2.176139224,5.010796927,8.326590261,13.815322022,25.389567339,49.66493532,81.24225136,97.985034316,100.0
Motorola_Edge -> Samsung_A52,H372,20.051627386,0.244140099,0.493295376,2.137075353,4.96522889,8.311248846,13.970834485,25.907680838,50.169060074,81.361975439,98.010950414,100.0
Motorola_Edge -> Samsung_A52,H405,102.937523487,0.403493015,3.699837563,10.367564103,22.407392179,44.643601071,74.455314104,92.851088518,98.133049129,99.81208377,100.0,100.0
Motorola_Edge -> Samsung_A52,H493,26.915604713,0.459525985,13.84014629,31.528531652,56.602018343,81.695883207,93.623722905,97.6698635,99.543243796,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H516,31.71323715,0.379243222,1.859453521,5.611857448,10.378430168,19.394650253,37.289973687,66.256813937,90.430712263,98.28246343,100.0,100.0
Motorola_Edge -> Samsung_A52,H549,78.506590591,0.40924461,4.130028315,11.640166668,25.578173771,48.864568862,78.059636971,93.776338256,98.217689214,99.821573074,100.0,100.0
Motorola_Edge -> Samsung_A52,H615,13.040486694,0.260516217,0.595757943,2.550813754,5.660565505,9.209064075,15.049537798,26.872667903,50.125207085,80.454952847,97.799876739,100.0
Motorola_Edge -> Samsung_A52,H616,41.595386411,0.455451946,6.877254622,16.248295469,33.39591741,60.31578164,85.090681299,95.386385995,98.949149353,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H617,18.952234005,0.419445292,5.78381327,14.803598952,31.733977043,58.05105324,84.185839957,95.784899706,98.993423031,100.0,100.0,100.0
Motorola_Edge -> Samsung_A52,H666,49.710833423,0.406012641,4.290560027,11.799589288,25.377271003,48.162063965,76.976170822,93.242156983,98.093322504,99.75640091,100.0,100.0
Motorola_Edge -> Samsung_A52,H668,62.976726278,0.423039377,5.443634147,14.759171215,31.886146483,58.379276443,84.420861616,95.641372839,99.018946334,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,F827,26.333632455,15.0335398,36.586208538,64.306453879,85.173308962,94.58768037,97.56101607,99.008236169,99.850140336,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,G190,50.804696706,0.471866112,21.625667008,43.30795007,73.373912881,92.556565596,97.235855275,99.101967827,100.0,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H030,29.3931356,0.450456242,12.35247949,31.253482229,57.442872694,83.298812627,94.510984105,98.181114383,99.627883801,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H037,119.926203974,0.455789721,14.537426286,36.039755323,67.488163538,89.51363846,96.392671544,98.732797811,99.834549778,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H038,148.547014855,0.444326445,10.990979145,29.883472145,57.947778588,84.788892545,95.180232679,98.395253838,99.724907348,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H126,14.684586587,0.454675968,12.994288372,31.385912018,56.974912664,83.540497475,95.270900465,98.352245102,99.668824304,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H181,12.085496925,0.362926225,1.400295217,3.798179811,7.385717065,11.930194279,21.378119566,41.286590669,70.519082946,93.012029242,99.649250027,100.0
Samsung_A52 -> Motorola_Edge,H183,12.763119749,0.35378074,1.292294569,3.365047202,6.759357864,10.992623948,18.907412931,35.539679746,63.783832805,90.123470153,99.537536564,100.0
Samsung_A52 -> Motorola_Edge,H367,32.238613497,0.31011913,1.103071573,3.81224974,7.661005577,13.309572946,25.448813902,49.311441304,79.291366603,95.931464729,99.775216273,100.0
Samsung_A52 -> Motorola_Edge,H368,13.040980104,0.26699327,0.70268284,2.821839712,6.232058889,10.14052844,16.892409367,30.683365526,57.255025909,86.616236507,98.575991963,100.0
Samsung_A52 -> Motorola_Edge,H371,10.358379542,0.272342534,0.667075583,2.543779112,5.701219361,9.506741703,16.307745215,30.893923813,58.868830931,87.963578513,99.00387482,100.0
Samsung_A52 -> Motorola_Edge,H372,33.061915014,0.244124271,0.493279547,1.992833066,4.592928179,7.624186723,12.177085665,21.747099139,42.584739521,72.875382231,95.216102827,100.0
Samsung_A52 -> Motorola_Edge,H405,48.896082184,0.446413048,9.904021524,25.629026479,49.375343198,78.450499918,93.608589365,97.903548934,99.540952909,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H493,87.735174623,28.815979472,51.258719761,77.789429241,91.469282484,96.118732926,98.333358312,99.587133851,100.0,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H516,51.607502367,0.400271432,2.481347018,7.248507133,13.721066147,27.707460986,51.977443435,80.721825418,95.166144516,99.318682732,100.0,100.0
Samsung_A52 -> Motorola_Edge,H549,10.717528521,0.466470346,17.522445359,37.26546211,65.876171314,87.670572796,95.895702702,98.463083391,99.713064443,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H615,20.081335788,0.26485694,0.627823914,2.420490007,5.293293068,8.402342693,13.140799348,22.712064855,42.640257269,72.050271732,94.555058532,100.0
Samsung_A52 -> Motorola_Edge,H616,73.145448235,0.498725542,14.969882461,33.215781135,59.361106864,83.946703309,94.636069724,98.179408737,99.730607166,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H617,47.432323747,0.460906349,17.641132421,37.449192083,65.491228739,87.475404843,96.326202735,98.78320003,99.884249411,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H666,43.689209584,0.471591674,23.192587736,44.560421,73.312581055,90.388119009,96.410692463,98.615327279,99.716420072,100.0,100.0,100.0
Samsung_A52 -> Motorola_Edge,H668,12.899936739,0.466503119,17.221452996,37.12546416,65.119618618,87.326091152,96.110504861,98.770356534,99.868608937,100.0,100.0,100.0
"""))
OUTLINES = {
    "Motorola_Edge_H615_02.jpg": [(80,130),(81,125),(83,122),(88,117),(91,112),
        (95,108),(100,104),(106,101),(110,100.5),(115,103),(121,105),(129,108.5),
        (126,112),(123,118),(120,122),(115,125),(111,129),(103,131),(95,133),
        (88,134.5),(83,134)],
    "Samsung_A52_H615_02.jpg": [(85,138),(85,134),(88,130),(93,125),(96,121),
        (99,117),(104,115),(108,111),(113,108.5),(117,109),(122,112),(129,114),
        (133,117),(130,121),(127,127),(124,130),(119,133),(115,137),(108,138.5),
        (102,140),(96,142),(89,142.5),(86,141)],
}

In [ ]:
def locate_inputs():
    required = set(boxes.index) | {"Training_labels_updated.csv"}
    names = {p.name:p for p in INPUT.rglob("*") if p.name in required and p.is_file()}
    for archive in INPUT.rglob("*.zip"):
        if set(names) == required:
            break
        with zipfile.ZipFile(archive) as z:
            for member in z.namelist():
                name = Path(member).name
                if name in required and name not in names:
                    target = TEMP / name
                    with z.open(member) as src, target.open("wb") as dest:
                        shutil.copyfileobj(src,dest)
                    names[name] = target
    missing = sorted(required - names.keys())
    if missing:
        raise FileNotFoundError(f"Add Input ile yarışmayı ekle. Eksik dosya sayısı {len(missing)}, örnek {missing[:3]}")
    return names

paths = locate_inputs()
labels = pd.read_csv(paths["Training_labels_updated.csv"],index_col="sample_id")[COLS]
assert len(boxes)==127 and boxes.index.is_unique and len(labels)==24
assert set(labels.index)==set(boxes.sample_id)
assert len(curves)==66 and not curves.duplicated(["exam","sample_id"]).any()
baseline = np.abs(curves[COLS].to_numpy()-labels.loc[curves.sample_id].to_numpy())[:,:-1] @ WIDTHS
np.testing.assert_allclose(baseline,curves.emd.to_numpy(),rtol=0,atol=1e-7)
print("127 fotoğraf, 24 toprak ve 66 saf v3 EMD doğrulandı.")

In [ ]:
weight_name = "IG2_full_set_cp_SAM"
weights = next(INPUT.rglob(weight_name),TEMP/weight_name)
if not weights.is_file():
    url = "https://zenodo.org/api/records/15728186/files/IG2_full_set_cp_SAM/content"
    temp_weights = weights.with_suffix(".part")
    with urlopen(url,timeout=120) as src,temp_weights.open("wb") as dest:
        shutil.copyfileobj(src,dest,length=1024*1024)
    temp_weights.replace(weights)
def md5sum(path):
    digest = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()
assert md5sum(weights)=="5a5b74bc70a63b6605afd50465240969", "ImageGrains ağırlık dosyası doğrulanamadı"
model = models.CellposeModel(gpu=True,pretrained_model=str(weights.resolve()),use_bfloat16=False)
assert model.device.type=="cuda" and Path(model.pretrained_model).resolve()==weights.resolve()

In [ ]:
CACHE = OUT / "maskeler_ig45_cp406"

def surface(box):
    with Image.open(paths[box.name]) as src:
        photo = ImageOps.exif_transpose(src).convert("RGB")
    w, h = photo.size
    portrait = h > w
    assert tuple(sorted((w,h), reverse=True)) == (box.width, box.height)
    ref, base = PHONES[box.phone]
    ppm = base * max(w,h) / ref
    rect = (w-box.bottom,box.left,w-box.top,box.right) if portrait else (
        box.left,box.top,box.right,box.bottom)
    roi = photo.crop(rect)
    dims = tuple(round(n * PPM / ppm) for n in roi.size)
    rgb = np.asarray(roi.resize(dims, Image.Resampling.LANCZOS))
    return rgb, portrait, ppm

def predict(model, name, rgb):
    CACHE.mkdir(parents=True, exist_ok=True)
    saved = CACHE / (name+".npz")
    if saved.exists():
        with np.load(saved) as data:
            mask = data["mask"]
    else:
        raw, *_ = model.eval(rgb,batch_size=1,bsize=256,flow_threshold=.4,cellprob_threshold=0.)
        assert raw.min() >= 0 and raw.max() <= 65535
        mask = raw.astype(np.uint16)
        tmp = saved.with_suffix(".tmp.npz")
        np.savez_compressed(tmp,mask=mask)
        tmp.replace(saved)
    assert mask.shape == rgb.shape[:2] and mask.dtype == np.uint16
    return mask

### İki telefonda H615 kontrolü

Tek bir otomatik sınırın elle çizilen sınırla kesişim/birleşim oranı (IoU) en az 0,50 olmalı. Aynı maskenin kısa ekseni 20 mm’yi aşmalı. Yerel CPU kontrolünde Motorola 0,915 ve Samsung 0,938 IoU verdi. GPU’da sayısal farklar olabilir. Sarı sınır elle çizileni, turkuaz en iyi otomatik eşleşmeyi gösterir.

In [ ]:
def check_candidate(name,outline,ax):
    box = boxes.loc[name]
    rgb,portrait,ppm = surface(box)
    mask = predict(model,name,rgb)
    if portrait:
        rgb,mask = np.rot90(rgb),np.rot90(mask)
    h,w = mask.shape
    scale = np.array([w/(box.right-box.left),h/(box.bottom-box.top)])*ppm
    pts = np.rint(np.array(outline)*scale-.5).astype(np.int32)
    ref = np.zeros(mask.shape,np.uint8)
    cv2.fillPoly(ref,[pts],1)
    overlaps = np.bincount(mask[ref>0],minlength=int(mask.max())+1)
    overlaps[0]=0
    areas=np.bincount(mask.ravel(),minlength=len(overlaps))
    ratios=overlaps/(areas+ref.sum()-overlaps)
    best=int(np.argmax(ratios))
    matched=(mask==best) if best else np.zeros_like(ref,dtype=bool)
    short=regionprops(matched.astype(np.uint8))[0].axis_minor_length/PPM if best else 0.
    ax.imshow(rgb)
    ax.plot(*np.vstack([pts,pts[0]]).T,color="gold")
    if matched.any():
        ax.contour(matched,levels=[.5],colors=["cyan"],linewidths=1)
    left,top=pts.min(axis=0)-35
    right,bottom=pts.max(axis=0)+35
    ax.set(xlim=(left,right),ylim=(bottom,top),title=f"{box.phone} · IoU {ratios[best]:.3f}")
    ax.axis("off")
    return dict(file=name,iou=ratios[best],kapsam=overlaps[best]/ref.sum(),kisa_eksen_mm=short)

fig,axes=plt.subplots(1,2,figsize=(9,4),layout="constrained")
audit=pd.DataFrame([check_candidate(name,outline,ax) for ax,(name,outline) in zip(axes,OUTLINES.items())])
fig.legend(handles=[Line2D([0],[0],color="gold",label="Elle çizilen"),Line2D([0],[0],color="cyan",label="Otomatik")],loc="lower center",ncol=2)
fig.savefig(OUT/"h615_gpu_kontrol.png",dpi=140)
plt.show()
audit.to_csv(OUT/"h615_gpu_kontrol.csv",index=False)
print(audit.to_string(index=False))
assert audit.iou.ge(.5).all() and audit.kisa_eksen_mm.gt(20).all(), "GPU maske kontrolü geçmedi. Alan düzeltmesini değerlendirmeden dur."

### Tam yüzey ölçümü

Kısa ekseni 20 mm’den büyük otomatik maskelerin toplam piksel alanını, toprak kadrajının tüm piksel alanına bölüyoruz. Tamamlanan maskeler çalışma klasörüne kaydedilir. Aynı oturumda hücre tekrar çalıştırılırsa biten fotoğraflar önbellekten okunur.

In [ ]:
records=[]
for i,(name,box) in enumerate(boxes.iterrows(),start=1):
    rgb,_,_=surface(box)
    mask=predict(model,name,rgb)
    stones=[p for p in regionprops(mask) if p.axis_minor_length > 20*PPM]
    area_pct=100*sum(p.area for p in stones)/mask.size
    records.append(dict(file=name,sample_id=box.sample_id,phone=box.phone,alan_pct=area_pct,aday_sayisi=len(stones)))
    if i%5==0 or i==len(boxes):
        print(f"{i}/{len(boxes)} fotoğraf",flush=True)
frame=pd.DataFrame(records)
assert len(frame)==127 and frame.file.is_unique and frame.alan_pct.between(0,100).all()
frame.to_csv(OUT/"fotograf_alanlari.csv",index=False)

### Saf v3 ve 20 mm düzeltmesi

Her turda bir toprağın bütün çekimleri ayrılır. Diğer toprakların fotoğraflarından ölçülen ortalama alan ile laboratuvardaki iri kütle payı arasında, başlangıcı sıfır olan negatif olmayan tek çarpan öğrenilir. Hedef fotoğraflarda alan sıfırsa saf v3 korunur. Alan pozitifse 20 mm düzeltmesi komşu 6,3 ve 63 mm geçiş yüzdeleri arasında tutulur. Diğer on elek aynıdır. Telefonlar arası sınamada öğrenme yalnız kaynak telefonla yapılır.

In [ ]:
def evaluate(frame,curves,labels,cols,widths):
    records = []
    idx = cols.index("20")
    for exam,block in curves.groupby("exam"):
        src,dst = (frame,frame) if exam=="24" else (
            frame[frame.phone==exam.split(" -> ")[0]],frame[frame.phone==exam.split(" -> ")[1]])
        train_area = src.groupby("sample_id").alan_pct.mean().loc[block.sample_id]
        test_area = dst.groupby("sample_id").alan_pct.mean().loc[block.sample_id]
        for sample,base in block.set_index("sample_id").iterrows():
            train = train_area.drop(sample)
            y = 100-labels.loc[train.index,"20"].to_numpy()
            x = train.to_numpy()
            denominator = x @ x
            factor = max(0., x @ y / denominator) if denominator>0 else np.nan
            curve = base[cols].to_numpy(dtype=float)
            actual = labels.loc[sample,cols].to_numpy()
            np.testing.assert_allclose(np.abs(actual-curve)[:-1] @ widths,base.emd,atol=1e-7,rtol=0)
            corrected = curve.copy()
            a = test_area.loc[sample]
            if a>0 and np.isfinite(factor):
                corrected[idx] = np.clip(100-factor*a,curve[idx-1],curve[idx+1])
            assert (np.diff(corrected)>=-1e-8).all()
            records.append(dict(exam=exam,sample_id=sample,alan_pct=a,carpan=factor,
                ref_sayisi=len(train),gercek_iri=100-actual[idx],v3_iri=100-curve[idx],
                duzeltilmis_iri=100-corrected[idx],v3_mae20=abs(curve[idx]-actual[idx]),
                duzeltilmis_mae20=abs(corrected[idx]-actual[idx]),v3_emd=base.emd,
                duzeltilmis_emd=np.abs(actual-corrected)[:-1] @ widths))
    result = pd.DataFrame(records)
    assert len(result)==66 and not result.duplicated(["exam","sample_id"]).any()
    difference = result.v3_emd-result.duzeltilmis_emd
    np.testing.assert_allclose(difference,(result.v3_mae20-result.duzeltilmis_mae20)*widths[idx],atol=1e-7,rtol=0)
    return result

def summarize(result):
    groups = list(result.groupby("exam"))+[("22",result[(result.exam=="24") & ~result.sample_id.isin(["H374","H615"])])]
    summaries = []
    for name,g in groups:
        gain = g.v3_emd-g.duzeltilmis_emd
        loo = (gain.sum()-gain)/(len(g)-1)
        summaries.append(dict(sinama=name,n=len(g),**g[["v3_mae20","duzeltilmis_mae20","v3_emd","duzeltilmis_emd"]].mean(),
                              ortalama_kazanc=gain.mean(),ortanca_kazanc=gain.median(),iyilesen=sum(gain>1e-7),
                              ayni=sum(abs(gain)<=1e-7),kotulesen=sum(gain < -1e-7),
                              tek_puan_haric_min=loo.min(),tek_puan_haric_max=loo.max()))
    return pd.DataFrame(summaries)

In [ ]:
result=evaluate(frame,curves,labels,COLS,WIDTHS)
summary=summarize(result)
assert sorted(summary.n.tolist())==[21,21,22,24]
result.to_csv(OUT/"toprak_sonuclari.csv",index=False,float_format="%.9f")
summary.to_csv(OUT/"ozet.csv",index=False,float_format="%.9f")
print(summary.round(4).to_string(index=False))
print("22 satırı, H374 ve H615’in puanları ortalamadan çıkarılınca kalan 22 toprağı gösterir.")
print("20 mm mutlak hatası yüzde puandır. tek_puan_haric sütunları tek sonuç çıkarıldığındaki ortalama EMD kazancıdır.")
print("Dosyalar:",OUT)